# A/B Test 03 — Average Order Value (AOV)

**Question.** Did the treatment change the average value of an order?

**Key modeling decision.** AOV is value **per order**, so it is computed over **purchasers only** (`purchased == 1`). Including non-buyers (whose `order_value` is 0) would drag the mean down and turn the metric into ARPU instead.

**Metric type.** `order_value` is continuous → compare **means** → **Welch's t-test** (`equal_var=False`).


In [1]:
import pandas as pd
from scipy import stats

df = pd.read_parquet("../data/ab_test_data.parquet")
df.head()

,user_id,group,clicked,purchased,order_value,returned_30d
0,T01163,treatment,1,0,0.00,1
1,T04385,treatment,0,0,0.00,1
2,C01902,control,1,1,96.31,0
3,C03397,control,0,0,0.00,0
4,T05695,treatment,1,0,0.00,1


In [2]:
# Data quality checks (run before trusting any metric)
print("shape:", df.shape)
print("\nnulls:\n", df.isnull().sum())
print("\nduplicate user_id:", df["user_id"].duplicated().sum())

# SRM (sample ratio mismatch): groups should be ~50/50
print("\ngroup sizes:\n", df["group"].value_counts())

# Binary columns must contain only 0/1
for col in ["clicked", "purchased", "returned_30d"]:
    print(col, "unique:", sorted(df[col].unique()))

shape: (12000, 6)

nulls:
 user_id         0
group           0
clicked         0
purchased       0
order_value     0
returned_30d    0
dtype: int64

duplicate user_id: 0

group sizes:
 group
treatment    6000
control      6000
Name: count, dtype: int64
clicked unique: [np.int64(0), np.int64(1)]
purchased unique: [np.int64(0), np.int64(1)]
returned_30d unique: [np.int64(0), np.int64(1)]


## Filter to purchasers, then compute AOV

In [3]:
buyers = df[df["purchased"] == 1]
print("buyers per group:\n", buyers["group"].value_counts())
print("min order_value among buyers:", buyers["order_value"].min())  # should be > 0

print(buyers.groupby("group")["order_value"].agg(["mean", "median", "std", "count"]))

buyers per group:
 group
treatment    551
control      327
Name: count, dtype: int64
min order_value among buyers: 5.0
                mean  median        std  count
group                                         
control    86.119878   84.64  23.840802    327
treatment  86.028094   86.68  25.122008    551


## Statistical test — Welch's t-test

`equal_var=False` runs Welch's t-test, which does **not** assume equal variances. It is a safe default: when variances are equal it matches Student's t-test, and when they are not it stays correct.

In [4]:
a = buyers[buyers["group"] == "control"]["order_value"]
b = buyers[buyers["group"] == "treatment"]["order_value"]

stat, pval = stats.ttest_ind(a, b, equal_var=False)
print(f"control AOV = {a.mean():.2f} | treatment AOV = {b.mean():.2f}")
print(f"p-value = {pval:.4f}")
print("Significant" if pval < 0.05 else "Not significant")

control AOV = 86.12 | treatment AOV = 86.03
p-value = 0.9569
Not significant


## Result

AOV is **86.12 (control)** vs **86.03 (treatment)**, p ≈ 0.96 → **not significant**. Each buyer spends essentially the same in both groups.

This is a clean null: the estimates are nearly identical *and* p is near 1, so the data genuinely supports "no difference" (not an underpowered miss).

## Concepts

### AOV vs ARPU (the most common trap)
| | AOV | ARPU |
|---|---|---|
| Definition | Average value **per order** | Average revenue **per user** |
| Filter | `purchased == 1` (buyers only) | none (everyone, non-buyers = 0) |
| Captures | Spend size only | Conversion **and** spend |
| Relationship | — | **ARPU ≈ conversion rate × AOV** |

A revenue question ("did we make more money?") should be answered with **ARPU / total revenue**, not AOV — AOV ignores how many people bought.

### Why Welch's t-test?
The groups have different standard deviations (23.8 vs 25.1) and unequal sizes (327 vs 551). Welch handles both safely, so it is the default here.

### Reading a non-significant result
p > 0.05 means "we cannot reject the null", **not** "the groups are identical". Here, because the means are also nearly equal, it is a true null. If the means had been far apart with a high p-value, that would instead suggest low statistical power.
